In [0]:
"""
Modelo de Machine Learning - Previsão de Risco de Bandeira Vermelha
Arquitetura: Ingestão de Dados (ANEEL + Open-Meteo) + Classificador Random Forest
"""

import os
import time
import requests
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Caminhos absolutos do Databricks
PASTA_SAIDA = "/Workspace/Users/Groups/mba/MBA_Eng_Dados_TurmaG_Energia_Solar/src/dados_tratados"
ARQUIVO_CIDADES = f"{PASTA_SAIDA}/cidades_com_usina_hidreletrica.csv"

TOP_N_CIDADES = 15 

def buscar_coordenadas(municipio: str, uf: str):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={municipio}&count=1&language=pt&format=json"
    try:
        resp = requests.get(url).json()
        if "results" in resp and len(resp["results"]) > 0:
            return resp["results"][0]["latitude"], resp["results"][0]["longitude"]
    except: 
        pass
    return None, None

def extrair_dados_climaticos(lat: float, lon: float):
    # Previsão 16 dias
    url_prev = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=precipitation_sum&timezone=America/Sao_Paulo&forecast_days=16"
    # Histórico de referência (Setembro 2025)
    url_hist = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2025-09-01&end_date=2025-09-30&daily=precipitation_sum&timezone=America/Sao_Paulo"
    
    chuva_prevista = 0.0
    chuva_historica = 0.0
    
    try:
        r_prev = requests.get(url_prev).json()
        if "daily" in r_prev and "precipitation_sum" in r_prev["daily"]:
            chuva_prevista = sum([c for c in r_prev["daily"]["precipitation_sum"] if c is not None])
            
        r_hist = requests.get(url_hist).json()
        if "daily" in r_hist and "precipitation_sum" in r_hist["daily"]:
            chuva_historica = sum([c for c in r_hist["daily"]["precipitation_sum"] if c is not None])
    except:
        pass
        
    return chuva_prevista, chuva_historica

def criar_base_treino_historica():
    """
    Gera um dataset de treino simulado mas fisicamente coerente para 
    ensinar o Random Forest a detectar bandeira vermelha com base em seca.
    """
    np.random.seed(42)
    n_amostras = 500
    
    potencia = np.random.uniform(10000, 1500000, n_amostras)
    chuva_prev = np.random.uniform(0, 150, n_amostras)
    chuva_hist = np.random.uniform(50, 200, n_amostras)
    razao = chuva_prev / (chuva_hist + 1e-5)
    
    # Alvo (Target): 1 se a razão de chuva for baixa e a potência alta (crítico), 0 caso contrário
    alvo = np.where((razao < 0.35) & (potencia > 200000), 1, 0)
    
    df_treino = pd.DataFrame({
        'potencia': potencia,
        'chuva_prevista': chuva_prev,
        'chuva_historica': chuva_hist,
        'razao_seca': razao,
        'bandeira_vermelha': alvo
    })
    return df_treino

def main():
    if not os.path.exists(ARQUIVO_CIDADES):
        print("ERRO: Arquivo base de cidades não encontrado.")
        return

    print("1. Treinando o Modelo de Machine Learning (Random Forest)...")
    df_treino = criar_base_treino_historica()
    X = df_treino[['potencia', 'chuva_prevista', 'chuva_historica', 'razao_seca']]
    y = df_treino['bandeira_vermelha']
    
    modelo = RandomForestClassifier(n_estimators=100, random_state=42)
    modelo.fit(X, y)
    print("-> Modelo treinado com sucesso!")

    print("\n2. Coletando dados reais atuais (ANEEL + Open-Meteo)...")
    df = pd.read_csv(ARQUIVO_CIDADES)
    df = df.sort_values(by="PotenciaOutorgadaTotalKw", ascending=False).head(TOP_N_CIDADES)

    dados_iniciais = []
    for _, linha in df.iterrows():
        lat, lon = buscar_coordenadas(linha["Municipio"], linha["UF"])
        time.sleep(0.3)
        
        if lat and lon:
            prev, hist = extrair_dados_climaticos(lat, lon)
            time.sleep(0.3)
            
            dados_iniciais.append({
                'municipio': linha["Municipio"],
                'potencia': linha["PotenciaOutorgadaTotalKw"],
                'chuva_prevista': prev,
                'chuva_historica': hist,
                'razao_seca': prev / (hist + 1e-5)
            })

    df_predicao = pd.DataFrame(dados_iniciais)
    
    # 3. Aplicando o Modelo de Machine Learning para prever o risco em cada polo
    X_pred = df_predicao[['potencia', 'chuva_prevista', 'chuva_historica', 'razao_seca']]
    df_predicao['predicao_risco'] = modelo.predict(X_pred)

    # 4. Decisão Sistêmica Global
    polos_em_risco = df_predicao['predicao_risco'].sum()
    percentual_crítico = (polos_em_risco / len(df_predicao)) * 100

    print("\n" + "="*40)
    if percentual_crítico >= 25.0: # Se 25% ou mais dos principais polos sofrerem previsão crítica
        print("BANDEIRA VERMELHA OU ACIMA: SIM")
    else:
        print("BANDEIRA VERMELHA OU ACIMA: NÃO")
    print("="*40)
    print(f"-> Polos geradores principais avaliados: {len(df_predicao)}")
    print(f"-> Polos apontados pelo modelo com alerta de seca: {polos_em_risco}")

if __name__ == "__main__":
    main()